In [1]:
import requests
import boto3
import json

In [ ]:
# Last.fm API Credentials
API_KEY = '#####'          # From last.fm/api/account/create
USERNAME = 'xu7'         # Your Last.fm username
# xu7

# ─── AWS Config ───────────────────────────────────────────────────────────────
AWS_REGION = 'us-east-1'       # e.g. 'us-east-1'
EVENT_SOURCE = 'custom.lastfm'
EVENT_DETAIL_TYPE = 'LastFM API Trigger'

BASE_URL = "http://ws.audioscrobbler.com/2.0/"

In [3]:
# ─── Helper Function: Make API calls ─────────────────────────────────────────
# This function sends requests to Last.fm API and returns JSON response safely.

def lastfm_get(method, extra_params={}):
    params = {
        "method": method,
        "user": USERNAME,
        "api_key": API_KEY,
        "format": "json",
        **extra_params
    }

    response = requests.get(BASE_URL, params=params)
    print(f"[{method}] Status: {response.status_code}")

    if response.status_code == 200 and response.text.strip():
        return response.json()
    else:
        print(f"Error: {response.text}")
        return {}

In [4]:
# ─── 1. Get User Profile ─────────────────────────────────────────────────────
# Fetches basic user details like name, country, playcount, and registration info.

def get_user_info():
    print("\n=== Fetching User Profile ===")
    data = lastfm_get("user.getinfo")

    print(json.dumps(data, indent=4))

    if 'user' in data:
        user = data['user']
        print(f"Username: {user.get('name')}")
        print(f"Real Name: {user.get('realname')}")
        print(f"Country: {user.get('country')}")
        print(f"Total Scrobbles: {user.get('playcount')}")
        print(f"Registered: {user.get('registered', {}).get('#text')}")

    return data

In [5]:
# ─── 2. Get Recent Tracks ─────────────────────────────────────────────────────
# Fetches the user's most recently played tracks (listening history).

def get_recent_tracks(limit=10):
    print(f"\n=== Fetching Last {limit} Recent Tracks ===")
    data = lastfm_get("user.getrecenttracks", {"limit": limit})

    if 'recenttracks' in data:
        tracks = data['recenttracks'].get('track', [])

        for i, track in enumerate(tracks[:limit], 1):
            artist = track.get('artist', {}).get('#text', 'Unknown')
            name = track.get('name', 'Unknown')
            album = track.get('album', {}).get('#text', 'Unknown')

            print(f"{i}. {name} — {artist} [{album}]")

    return data

In [6]:
# ─── 3. Get Top Tracks ───────────────────────────────────────────────────────
# Returns most played tracks for a selected time period.

def get_top_tracks(period='overall', limit=10):
    """
    period options:
    '7day', '1month', '3month', '6month', '12month', 'overall'
    """
    print(f"\n=== Fetching Top {limit} Tracks ({period}) ===")
    data = lastfm_get("user.gettoptracks", {"period": period, "limit": limit})

    if 'toptracks' in data:
        tracks = data['toptracks'].get('track', [])

        for i, track in enumerate(tracks, 1):
            name = track.get('name', 'Unknown')
            artist = track.get('artist', {}).get('name', 'Unknown')
            playcount = track.get('playcount', 0)

            print(f"{i}. {name} — {artist} (played {playcount} times)")

    return data

In [7]:
# ─── 4. Get Top Artists ───────────────────────────────────────────────────────
# Fetches most listened artists based on play count.

def get_top_artists(period='overall', limit=10):
    print(f"\n=== Fetching Top {limit} Artists ({period}) ===")
    data = lastfm_get("user.gettopartists", {"period": period, "limit": limit})

    if 'topartists' in data:
        artists = data['topartists'].get('artist', [])

        for i, artist in enumerate(artists, 1):
            name = artist.get('name', 'Unknown')
            playcount = artist.get('playcount', 0)

            print(f"{i}. {name} (played {playcount} times)")

    return data

In [8]:
# ─── 5. Get Top Albums ───────────────────────────────────────────────────────
# Returns most played albums for the user.

def get_top_albums(period='overall', limit=10):
    print(f"\n=== Fetching Top {limit} Albums ({period}) ===")
    data = lastfm_get("user.gettopalbums", {"period": period, "limit": limit})

    if 'topalbums' in data:
        albums = data['topalbums'].get('album', [])

        for i, album in enumerate(albums, 1):
            name = album.get('name', 'Unknown')
            artist = album.get('artist', {}).get('name', 'Unknown')
            playcount = album.get('playcount', 0)

            print(f"{i}. {name} — {artist} (played {playcount} times)")

    return data

In [9]:
# ─── 6. Get Loved Tracks ─────────────────────────────────────────────────────
# Fetches tracks marked as "loved" by the user on Last.fm.

def get_loved_tracks(limit=10):
    print(f"\n=== Fetching Loved Tracks ===")
    data = lastfm_get("user.getlovedtracks", {"limit": limit})

    if 'lovedtracks' in data:
        tracks = data['lovedtracks'].get('track', [])

        for i, track in enumerate(tracks, 1):
            name = track.get('name', 'Unknown')
            artist = track.get('artist', {}).get('name', 'Unknown')

            print(f"{i}. {name} — {artist}")

    return data

In [ ]:
# ─── Send to AWS CloudWatch ─────────────────────────────────────────────────
# Sends collected Last.fm data to AWS EventBridge / CloudWatch.

def send_event_to_cloudwatch(data, detail_type=EVENT_DETAIL_TYPE):
    print("\n=== Sending to CloudWatch ===")

    client = boto3.client(
        'events',
        region_name=AWS_REGION,
        aws_access_key_id='####',
        aws_secret_access_key='#######'
    )

    response = client.put_events(
        Entries=[{
            'Source': EVENT_SOURCE,
            'DetailType': detail_type,
            'Detail': json.dumps(data),
            'EventBusName': 'default',
        }]
    )

    failed = response.get('FailedEntryCount', 0)

    if failed == 0:
        print("✅ Event sent to CloudWatch successfully!")
    else:
        print(f"❌ Failed entries: {response['Entries']}")

In [11]:
# ─── Main Pipeline ───────────────────────────────────────────────────────────
# Orchestrates full data pipeline: fetch Last.fm data and send to AWS.

def main():
    user_info     = get_user_info()
    recent_tracks = get_recent_tracks(limit=1)
    top_tracks    = get_top_tracks(period='1month', limit=10)
    top_artists   = get_top_artists(period='overall', limit=10)
    top_albums    = get_top_albums(period='overall', limit=10)
    loved_tracks  = get_loved_tracks(limit=10)

    all_data = {
        "user_info": user_info,
        "recent_tracks": recent_tracks,
        "top_tracks": top_tracks,
        "top_artists": top_artists,
        "top_albums": top_albums,
        "loved_tracks": loved_tracks
    }

    send_event_to_cloudwatch(all_data)
    print("\n✅ All done!")

if __name__ == "__main__":
    main()


=== Fetching User Profile ===
[user.getinfo] Status: 200
{
    "user": {
        "name": "xu7",
        "age": "0",
        "subscriber": "0",
        "realname": "",
        "bootstrap": "0",
        "playcount": "902174",
        "artist_count": "1",
        "playlists": "0",
        "track_count": "2",
        "album_count": "2",
        "image": [
            {
                "size": "small",
                "#text": "https://lastfm.freetls.fastly.net/i/u/34s/e832fd3adade4e1b60f5ce16ce62c089.png"
            },
            {
                "size": "medium",
                "#text": "https://lastfm.freetls.fastly.net/i/u/64s/e832fd3adade4e1b60f5ce16ce62c089.png"
            },
            {
                "size": "large",
                "#text": "https://lastfm.freetls.fastly.net/i/u/174s/e832fd3adade4e1b60f5ce16ce62c089.png"
            },
            {
                "size": "extralarge",
                "#text": "https://lastfm.freetls.fastly.net/i/u/300x300/e832fd3adade4e1

In [ ]:
############# Full Code
# ─── Helper: Make API calls ───────────────────────────────────────────────────
def lastfm_get(method, extra_params={}):
    params = {
        "method": method,
        "user": USERNAME,
        "api_key": API_KEY,
        "format": "json",
        **extra_params
    }
    response = requests.get(BASE_URL, params=params)
    print(f"[{method}] Status: {response.status_code}")

    if response.status_code == 200 and response.text.strip():
        return response.json()
    else:
        print(f"❌ Error: {response.text}")
        return {}

# ─── 1. Get User Profile ──────────────────────────────────────────────────────
def get_user_info():
    print("\n=== Fetching User Profile ===")
    data = lastfm_get("user.getinfo")
    print(json.dumps(data, indent=4))
    if 'user' in data:
        user = data['user']
        print(f"✅ Username     : {user.get('name')}")
        print(f"   Real Name    : {user.get('realname')}")
        print(f"   Country      : {user.get('country')}")
        print(f"   Total Scrobbles: {user.get('playcount')}")
        print(f"   Registered   : {user.get('registered', {}).get('#text')}")
    return data

# ─── 2. Get Recent Tracks (like a listening history) ─────────────────────────
def get_recent_tracks(limit=10):
    print(f"\n=== Fetching Last {limit} Recent Tracks ===")
    data = lastfm_get("user.getrecenttracks", {"limit": limit})
    if 'recenttracks' in data:
        tracks = data['recenttracks'].get('track', [])
        for i, track in enumerate(tracks[:limit], 1):
            artist = track.get('artist', {}).get('#text', 'Unknown')
            name = track.get('name', 'Unknown')
            album = track.get('album', {}).get('#text', 'Unknown')
            print(f"  {i}. {name} — {artist} [{album}]")
    return data

# ─── 3. Get Top Tracks ────────────────────────────────────────────────────────
def get_top_tracks(period='overall', limit=10):
    """
    period options:
    '7day', '1month', '3month', '6month', '12month', 'overall'
    """
    print(f"\n=== Fetching Top {limit} Tracks ({period}) ===")
    data = lastfm_get("user.gettoptracks", {"period": period, "limit": limit})
    if 'toptracks' in data:
        tracks = data['toptracks'].get('track', [])
        for i, track in enumerate(tracks, 1):
            name = track.get('name', 'Unknown')
            artist = track.get('artist', {}).get('name', 'Unknown')
            playcount = track.get('playcount', 0)
            print(f"  {i}. {name} — {artist} (played {playcount} times)")
    return data

# ─── 4. Get Top Artists ───────────────────────────────────────────────────────
def get_top_artists(period='overall', limit=10):
    print(f"\n=== Fetching Top {limit} Artists ({period}) ===")
    data = lastfm_get("user.gettopartists", {"period": period, "limit": limit})
    if 'topartists' in data:
        artists = data['topartists'].get('artist', [])
        for i, artist in enumerate(artists, 1):
            name = artist.get('name', 'Unknown')
            playcount = artist.get('playcount', 0)
            print(f"  {i}. {name} (played {playcount} times)")
    return data

# ─── 5. Get Top Albums ────────────────────────────────────────────────────────
def get_top_albums(period='overall', limit=10):
    print(f"\n=== Fetching Top {limit} Albums ({period}) ===")
    data = lastfm_get("user.gettopalbums", {"period": period, "limit": limit})
    if 'topalbums' in data:
        albums = data['topalbums'].get('album', [])
        for i, album in enumerate(albums, 1):
            name = album.get('name', 'Unknown')
            artist = album.get('artist', {}).get('name', 'Unknown')
            playcount = album.get('playcount', 0)
            print(f"  {i}. {name} — {artist} (played {playcount} times)")
    return data

# ─── 6. Get Loved Tracks ─────────────────────────────────────────────────────
def get_loved_tracks(limit=10):
    print(f"\n=== Fetching Loved Tracks ===")
    data = lastfm_get("user.getlovedtracks", {"limit": limit})
    if 'lovedtracks' in data:
        tracks = data['lovedtracks'].get('track', [])
        for i, track in enumerate(tracks, 1):
            name = track.get('name', 'Unknown')
            artist = track.get('artist', {}).get('name', 'Unknown')
            print(f"  {i}. {name} — {artist}")
    return data

# ─── Send to AWS CloudWatch ───────────────────────────────────────────────────
def send_event_to_cloudwatch(data, detail_type=EVENT_DETAIL_TYPE):
    print(f"\n=== Sending to CloudWatch ===")
    client = boto3.client('events', region_name=AWS_REGION, aws_access_key_id='', aws_secret_access_key='')
    response = client.put_events(
        Entries=[{
            'Source': EVENT_SOURCE,
            'DetailType': detail_type,
            'Detail': json.dumps(data),
            'EventBusName': 'default',
        }]
    )
    failed = response.get('FailedEntryCount', 0)
    if failed == 0:
        print("✅ Event sent to CloudWatch successfully!")
    else:
        print(f"❌ Failed entries: {response['Entries']}")

# ─── Main ─────────────────────────────────────────────────────────────────────
def main():
    # Collect all data
    user_info      = get_user_info()
    recent_tracks  = get_recent_tracks(limit=1)
    top_tracks     = get_top_tracks(period='1month', limit=10)
    top_artists    = get_top_artists(period='overall', limit=10)
    top_albums     = get_top_albums(period='overall', limit=10)
    loved_tracks   = get_loved_tracks(limit=10)

    # Bundle everything together
    all_data = {
        "user_info"     : user_info,
        "recent_tracks" : recent_tracks,
        "top_tracks"    : top_tracks,
        "top_artists"   : top_artists,
        "top_albums"    : top_albums,
        "loved_tracks"  : loved_tracks
    }

    # Send bundled data to CloudWatch
    send_event_to_cloudwatch(all_data)
    print("\n✅ All done!")

if __name__ == "__main__":
    main()